# 02 · Cleaning + spec enrichment + simulated OBD-II features

Runs the pooling + enrichment contract from `ml.ingest` and the engineered features from `ml.features` (both tested). Order: clean cars-spec -> harmonise the three price datasets -> pool -> enrich -> engineer -> write `data/interim/merc_engineered.csv`.

In [ ]:
import sys, pathlib
for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_c / 'app').exists() and (_c / 'ml').exists():
        sys.path.insert(0, str(_c)); break
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from ml import ingest
sns.set_theme()

### Step 1 — cars-spec lookup (cleaned first)

In [ ]:
specs = ingest.clean_spec(); print('spec variants:', specs.shape); specs.head()

### Step 2 — harmonise + pool the three price datasets
Each source -> common schema, FX->RM, `source_market`. `clean_pool` drops nulls/outliers/dupes and adds `age`.

In [ ]:
pool = ingest.build_pool()
print('pooled rows:', len(pool), '| by market:', pool['source_market'].value_counts().to_dict())
pool.head()

### Step 3 — enrich every row with specs (class x year x nearest displacement)
`engine_size` is backfilled from the matched spec; `match_level` records join quality.

In [ ]:
enr = ingest.enrich(pool, specs)
print('match-level:', enr['match_level'].value_counts().to_dict())
print('engine_size null after backfill:', int(enr['engine_size'].isna().sum()))
enr[['model_class','variant','source_market','age','mileage','engine_size','torque_nm','top_speed_kmh','price_rm']].head()

### Step 4 — engineer the simulated OBD-II features + write the CSV

In [ ]:
df = ingest.build_engineered_csv()   # pool + enrich + engineer + write
print('final:', df.shape); df.head()

### `battery_soh` — deterministic base curves (from `ml.features`)
Time-dominant for petrol/diesel; non-linear in mileage for hybrids (mileage now in km).

In [ ]:
from ml.features import battery_soh_base, SOH_FLOOR
ages = np.arange(0, 21)
for ft in ['Petrol','Diesel','Hybrid']:
    plt.plot(ages, [battery_soh_base(ft, a, 100_000) for a in ages], marker='.', label=ft)
plt.axhline(SOH_FLOOR, color='gray', ls='--'); plt.legend()
plt.xlabel('age'); plt.ylabel('battery_soh_base (%)'); plt.title('@100,000 km'); plt.show()

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(11,4))
df['battery_soh'].plot.hist(bins=40, ax=ax[0], title='battery_soh (%)')
sns.scatterplot(data=df.sample(min(3000,len(df)), random_state=0), x='mileage', y='battery_soh',
                hue='fuel_type', s=8, ax=ax[1]); ax[1].set_title('SoH vs mileage'); plt.tight_layout()

### `trans_adapt_offset` — Manual is exactly 0; automatics strictly negative, scale with mileage

In [ ]:
print(df.groupby('transmission')['trans_adapt_offset'].agg(['mean','min','max']))
auto = df[df['transmission'] != 'Manual'].sample(min(3000,len(df)), random_state=0)
sns.scatterplot(data=auto, x='mileage', y='trans_adapt_offset', s=8)
plt.title('offset vs mileage (non-manual)'); plt.show()

### Enrichment quality by source
UK rows are displacement-refined (real engine size); germany/US use badge hint -> nearest real variant, else class-year representative.

In [ ]:
display(df.groupby(['source_market','match_level']).size().unstack(fill_value=0))
print('price_rm by market (median):', df.groupby('source_market')['price_rm'].median().to_dict())

Engineered dataset written to `data/interim/merc_engineered.csv` — the pooled model trains on this.